<a href="https://colab.research.google.com/github/mikerich135/credit-risk-modeling/blob/xgboost_by_richard/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost

In [1]:
# install dependencies
!pip install xgboost tensorflow google

In [2]:
"""
Credit Default Prediction
Models: XGBoost  →  ANN (Keras/TensorFlow)
Target: 1 = payment difficulties, 0 = no issues
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, classification_report, f1_score,
    RocCurveDisplay, PrecisionRecallDisplay,
)

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from google.colab import drive




In [3]:
# LOad DATA

rootpath = drive.mount('/content/drive')


#DATA_PATH = (
#    r"C:\Users\micha\OneDrive\Documents\Claude\loan-defaulter"
#    r"\DataMerge-20260510T015452Z-3-001\DataMerge\df_main\df_main.xls"
#)

DATA_PATH = "/content/drive/MyDrive/Project#24/data-kaggle/df_main/df_main_clean.xls"

OUT_DIR      = "/content/drive/MyDrive/Project#24/data-kaggle/df_main/output"

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


print("\n[1] Loading data...")
df = pd.read_csv(DATA_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[1] Loading data...


In [4]:
#Segregate the data
# Seperate train and test data

train_df = df[df["is_train"] == 1].copy()
test_df  = df[df["is_train"] == 0].copy()

DROP_COLS    = ["SK_ID_CURR", "TARGET", "is_train"] # Drop columns
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS] # Feature Columns

X_all   = train_df[FEATURE_COLS]
y_all   = train_df["TARGET"].astype(int)
X_test  = test_df[FEATURE_COLS]




In [5]:
#80/20 Train - Validate Split

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE
)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"  Train : {X_train.shape[0]:,} rows  |  Val: {X_val.shape[0]:,} rows")
print(f"  Features : {len(FEATURE_COLS)}")
print(f"  Class balance — 0: {(y_train==0).sum():,}  1: {(y_train==1).sum():,}"
      f"  (pos_weight ≈ {pos_weight:.1f})")


  Train : 246,008 rows  |  Val: 61,503 rows
  Features : 270
  Class balance — 0: 226,148  1: 19,860  (pos_weight ≈ 11.4)


In [6]:
#Preprocessing

# separate category columns and numeric columns
CAT_COLS = X_train.select_dtypes(include="object").columns.tolist()
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

# Create pipeline by Filling numberic values with median
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Create pipeline by Filling the missing value with most common value
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

# Combine both the pipeline
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUM_COLS),
    ("cat", categorical_pipe, CAT_COLS),
], remainder="drop")


# Fitting and Transforming
print("\n[2] Fitting preprocessor...")
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)
X_all_proc   = preprocessor.transform(X_all)

n_features = X_train_proc.shape[1]
print(f"  Processed feature matrix: {X_train_proc.shape}")



[2] Fitting preprocessor...
  Processed feature matrix: (246008, 270)


In [7]:
#XGBoost Implementation
#Configure the model
xgb_model = xgb.XGBClassifier(
    n_estimators       = 1000,
    learning_rate      = 0.05,
    max_depth          = 6,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    scale_pos_weight   = pos_weight,      # handles class imbalance
    eval_metric        = "auc",
    early_stopping_rounds = 50,
    use_label_encoder  = False,
    random_state       = RANDOM_STATE,
    n_jobs             = -1,
    verbosity          = 1,
)

#Train the model
print("\n[3] Training XGBoost (early stopping on val AUC)...")
xgb_model.fit(
    X_train_proc, y_train,
    eval_set=[(X_val_proc, y_val)],
    verbose=100,
)

xgb_val_proba = xgb_model.predict_proba(X_val_proc)[:, 1]
xgb_auc       = roc_auc_score(y_val, xgb_val_proba)

# Optimise threshold by F1
thresholds = np.linspace(0.05, 0.95, 100)
f1s = [f1_score(y_val, (xgb_val_proba >= t).astype(int), zero_division=0) for t in thresholds]
xgb_best_thresh = thresholds[np.argmax(f1s)]
xgb_pred = (xgb_val_proba >= xgb_best_thresh).astype(int)

print(f"\n  XGBoost — ROC-AUC : {xgb_auc:.4f}")
print(f"  Best threshold    : {xgb_best_thresh:.2f}  (F1-optimised)")
print(classification_report(y_val, xgb_pred,
                             target_names=["No Default", "Default"], digits=3))

# Feature importance (top 100)
feature_names = NUM_COLS + CAT_COLS
xgb_imp = pd.DataFrame({
    "feature":    feature_names,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False).head(100)

print("  Top 100 XGBoost features:")
print(xgb_imp.head(100).to_string(index=False))



[3] Training XGBoost (early stopping on val AUC)...
[0]	validation_0-auc:0.68863
[100]	validation_0-auc:0.77058
[200]	validation_0-auc:0.77760
[300]	validation_0-auc:0.77964
[400]	validation_0-auc:0.77989
[417]	validation_0-auc:0.77988

  XGBoost — ROC-AUC : 0.7802
  Best threshold    : 0.67  (F1-optimised)
              precision    recall  f1-score   support

  No Default      0.946     0.909     0.927     56538
     Default      0.285     0.414     0.337      4965

    accuracy                          0.869     61503
   macro avg      0.616     0.661     0.632     61503
weighted avg      0.893     0.869     0.879     61503

  Top 100 XGBoost features:
                                                  feature  importance
                                             EXT_SOURCE_3    0.032430
                                             EXT_SOURCE_2    0.029351
                                              CODE_GENDER    0.020842
                                      NAME_EDUCATION_TY

In [8]:
#ANN

def build_ann(input_dim: int, dropout_rate: float = 0.3) -> keras.Model:
    """3-hidden-layer feed-forward network with BatchNorm & Dropout."""
    inp = keras.Input(shape=(input_dim,), name="input")

    x = layers.Dense(512, kernel_initializer="he_normal")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(256, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(128, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout_rate)(x)

    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],
    )
    return model


ann_model = build_ann(n_features)
ann_model.summary()

# Class weights dict for Keras
class_weight_dict = {0: 1.0, 1: pos_weight}

ann_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_auc", mode="max",
        patience=10, restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_auc", mode="max",
        factor=0.5, patience=5, min_lr=1e-6, verbose=1,
    ),
]

print("\n[4] Training ANN...")
history = ann_model.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=100,
    batch_size=2048,
    class_weight=class_weight_dict,
    callbacks=ann_callbacks,
    verbose=1,
)

ann_val_proba = ann_model.predict(X_val_proc, batch_size=4096).ravel()
ann_auc       = roc_auc_score(y_val, ann_val_proba)

f1s = [f1_score(y_val, (ann_val_proba >= t).astype(int), zero_division=0) for t in thresholds]
ann_best_thresh = thresholds[np.argmax(f1s)]
ann_pred = (ann_val_proba >= ann_best_thresh).astype(int)

print(f"\n  ANN — ROC-AUC : {ann_auc:.4f}")
print(f"  Best threshold : {ann_best_thresh:.2f}  (F1-optimised)")
print(classification_report(y_val, ann_pred,
                             target_names=["No Default", "Default"], digits=3))



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 270)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       138,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 306,689 (1.17 MB)

 Trainable params: 304,897 (1.16 MB)

 Non-trainable params: 1,792 (7.00 KB)


[4] Training ANN...
Epoch 1/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - auc: 0.6680 - loss: 1.2293 - val_auc: 0.7403 - val_loss: 0.7323 - learning_rate: 0.0010
Epoch 2/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.7307 - loss: 1.1270 - val_auc: 0.7572 - val_loss: 0.6465 - learning_rate: 0.0010
Epoch 3/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.7468 - loss: 1.0976 - val_auc: 0.7610 - val_loss: 0.6297 - learning_rate: 0.0010
Epoch 4/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.7543 - loss: 1.0846 - val_auc: 0.7636 - val_loss: 0.6233 - learning_rate: 0.0010
Epoch 5/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.7599 - loss: 1.0744 - val_auc: 0.7648 - val_loss: 0.5768 - learning_rate: 0.0010
Epoch 6/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.7644 - loss: 1.0662 - val_auc: 0.7653 - val_loss: 0.5966 - learning_rate: 0.0010
Epoch 7/100
121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.7679 - loss: 1.0600 - val_auc: 0.7657 - val_loss: 0.

In [9]:
#Comparison Plot

print("\n[5] Generating plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("XGBoost vs ANN — Validation Set", fontsize=14, fontweight="bold")

# Plot 1 — AUC bar
ax = axes[0]
model_names = ["XGBoost", "ANN"]
aucs        = [xgb_auc, ann_auc]
colors      = ["#1f77b4", "#ff7f0e"]
bars = ax.bar(model_names, aucs, color=colors, width=0.4)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("ROC-AUC")
ax.set_title("ROC-AUC Comparison")
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontweight="bold")

# Plot 2 — ROC curves
ax = axes[1]
RocCurveDisplay.from_predictions(y_val, xgb_val_proba, ax=ax, name=f"XGBoost (AUC={xgb_auc:.4f})", color=colors[0])
RocCurveDisplay.from_predictions(y_val, ann_val_proba,  ax=ax, name=f"ANN     (AUC={ann_auc:.4f})",  color=colors[1])
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("ROC Curves")

# Plot 3 — ANN training history
ax = axes[2]
ax.plot(history.history["auc"],     label="Train AUC", color=colors[1])
ax.plot(history.history["val_auc"], label="Val AUC",   color=colors[1], linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("AUC")
ax.set_title("ANN Training History")
ax.legend()

plt.tight_layout()
fig.savefig(f"{OUT_DIR}\\model_comparison.png", dpi=150, bbox_inches="tight")
print("  Saved: model_comparison.png")

# Plot 4 — XGBoost feature importance
fig2, ax2 = plt.subplots(figsize=(9, 8))
ax2.barh(xgb_imp["feature"][::-1], xgb_imp["importance"][::-1], color="#1f77b4")
ax2.set_xlabel("XGBoost Feature Importance (gain)")
ax2.set_title("Top 20 Features — XGBoost")
plt.tight_layout()
fig2.savefig(f"{OUT_DIR}\\xgb_feature_importance.png", dpi=150, bbox_inches="tight")
print("  Saved: xgb_feature_importance.png")



[5] Generating plots...
  Saved: model_comparison.png
  Saved: xgb_feature_importance.png


In [10]:
# Test Prediction
print("\n[6] Generating test predictions...")

xgb_test_proba = xgb_model.predict_proba(X_test_proc)[:, 1]
ann_test_proba = ann_model.predict(X_test_proc, batch_size=4096).ravel()

# Ensemble: simple average of both model probabilities
ensemble_proba = (xgb_test_proba + ann_test_proba) / 2

submission = pd.DataFrame({
    "SK_ID_CURR":     test_df["SK_ID_CURR"].values,
    "TARGET_XGB":     xgb_test_proba,
    "TARGET_ANN":     ann_test_proba,
    "TARGET_ENSEMBLE": ensemble_proba,
})
submission.to_csv(f"{OUT_DIR}\\submission.csv", index=False)
print(f"  Saved: submission.csv  ({len(submission):,} rows)")

# ── Summary ───────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  {'Model':<18}  {'ROC-AUC':>9}  {'Threshold':>10}")
print(f"  {'-'*40}")
print(f"  {'XGBoost':<18}  {xgb_auc:>9.4f}  {xgb_best_thresh:>10.2f}")
print(f"  {'ANN':<18}  {ann_auc:>9.4f}  {ann_best_thresh:>10.2f}")
print("=" * 60)
print("\nDone.")



[6] Generating test predictions...
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
  Saved: submission.csv  (48,744 rows)

  FINAL SUMMARY
  Model                 ROC-AUC   Threshold
  ----------------------------------------
  XGBoost                0.7802        0.67
  ANN                    0.7671        0.70

Done.
